<a href="https://colab.research.google.com/github/cmd0850/machine_lerning/blob/main/%D0%94%D0%B6%D0%B5%D0%BD%D1%87%D0%B0%D0%BA%D0%BE_%D0%9B%D0%A0_9_%D0%A0%D0%B5%D0%BA%D0%BE%D0%BC%D0%B5%D0%BD%D0%B4%D0%B0%D1%86%D1%96%D0%B9%D0%BD%D1%96_%D1%81%D0%B8%D1%81%D1%82%D0%B5%D0%BC%D0%B8_%D0%A8%D0%B0%D0%B1%D0%BB%D0%BE%D0%BD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторна робота № 9
## Рекомендаційні системи: SVD та KNNBasic

**ПІБ студента:** __________________________  
**Група:** _________________________________  
**Дата:** __________________________________

> **Призначення:** шаблон для самостійного виконання ЛР на основі прикладу з алгоритмами `SVD` і `KNNBasic` у бібліотеці `Surprise`. Комірки з позначкою **TODO** потрібно заповнити власним кодом. Це не готовий розв’язок і не заміна офіційної умови роботи.

**Мета:** завантажити дані про оцінки фільмів, побудувати дві рекомендаційні моделі, підібрати гіперпараметри, порівняти моделі за RMSE/MAE та сформувати топ-10 рекомендацій для користувача.

**Дані:** вбудований набір `MovieLens 100K` (`ml-100k`) із бібліотеки `Surprise`.

**Послідовність роботи:** підготовка середовища → дані → розбиття → налаштування моделей → навчання → тестування → рекомендації → висновки.

## 1. Підготовка середовища

Комбінація бібліотек для **Google Colab із Python 3.13**. Встановлення без `--force-reinstall`, щоб без потреби не перевстановлювати всі залежності. Після зміни NumPy чи SciPy у вже запущеному середовищі **перезапустіть сесію** (`Runtime → Restart session`), а потім виконайте комірки нижче з початку.

In [1]:
%pip install -q \
    numpy==2.2.6 \
    scipy==1.15.3 \
    scikit-learn==1.6.1 \
    pandas==2.2.3 \
    scikit-surprise==1.1.5

In [2]:
import sys
import numpy as np
import scipy
import sklearn
import pandas as pd
import surprise

from surprise import Dataset, Reader, SVD, KNNBasic, accuracy
from surprise.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split as sklearn_train_test_split

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('scikit-learn:', sklearn.__version__)
print('pandas:', pd.__version__)
print('Surprise:', surprise.__version__)

Python: 3.13.18
NumPy: 2.4.6
SciPy: 1.15.3
scikit-learn: 1.6.1
pandas: 2.2.3
Surprise: 1.1.5


## 2. Завантаження та первинний аналіз даних

1. Завантажте вбудований набір `ml-100k`.
2. Перетворіть оцінки у `pandas.DataFrame` з колонками `user_id`, `item_id`, `rating`, `timestamp`.
3. Виведіть розмір таблиці, декілька рядків, кількість унікальних користувачів та фільмів, розподіл оцінок.

**Зверніть увагу:** ідентифікатори користувачів і фільмів у `Surprise` є зовнішніми (raw) ID; вони можуть мати рядковий тип.

In [3]:
# Завантаження даних (готова частина шаблону)
data = Dataset.load_builtin('ml-100k', prompt=False)

ratings_df = pd.DataFrame(
    data.raw_ratings,
    columns=['user_id', 'item_id', 'rating', 'timestamp']
)

print('Розмір:', ratings_df.shape)
display(ratings_df.head())

Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to /root/.surprise_data/ml-100k
Розмір: (100000, 4)


,user_id,item_id,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


In [4]:
# TODO 2.1 — Дослідіть набір даних:
# - кількість унікальних користувачів;
# - кількість унікальних фільмів;
# - мінімальне, максимальне, середнє значення rating;
# - пропущені значення (за наявності).

# Ваш код:
print(ratings_df['item_id'].unique().shape)
print(ratings_df['user_id'].unique().shape)
print(ratings_df['rating'].max(), ratings_df['rating'].min(), ratings_df['rating'].mean())
print(ratings_df['rating'].isna().sum())

(1682,)
(943,)
5.0 1.0 3.52986
0


**Короткі спостереження щодо даних:**

_Запишіть 2–3 речення про кількість користувачів, фільмів та характер оцінок._

## 3. Розбиття на навчальну та тестову вибірки

Виконайте випадкове розбиття **80% / 20%** із `random_state=42` за прикладом. Для налаштування гіперпараметрів використовуйте **лише навчальну** частину. Тестова вибірка має залишатися недоторканою до фінальної оцінки.

In [5]:
# TODO 3.1 — Створіть train_df та test_df із ratings_df.
# Використайте sklearn_train_test_split(..., test_size=0.2, random_state=42).

# Ваш код:
train_df, test_df = sklearn_train_test_split(ratings_df, test_size=0.2, random_state=42)

In [6]:
# TODO 3.2 — Підготуйте об’єкти для Surprise:
# 1. reader = Reader(rating_scale=...)
#    Підказка: data.reader.rating_scale
# 2. train_data = Dataset.load_from_df(...)
#    Візьміть тільки user_id, item_id, rating із train_df.
# 3. testset = список кортежів (user_id, item_id, rating) із test_df.
# 4. Перевірте розміри train/test.

# Ваш код:
print(data.reader.rating_scale)
reader = Reader(rating_scale=(1, 5))

train_data = Dataset.load_from_df(train_df[['user_id', 'item_id', 'rating']], reader)
testset=list(test_df[['user_id', 'item_id', 'rating']].itertuples(index=False, name=None))

(1, 5)


## 4. Гіперпараметри моделей

Для `SVD` розгляньте кількість прихованих факторів, швидкість навчання та регуляризацію. Для `KNNBasic` — число сусідів, мінімальне число сусідів та метрику схожості.

Нижче — **початкова сітка** з аналізованого прикладу. Її можна змінити й обґрунтувати вибір.

In [8]:
# Стартові сітки параметрів (можна змінювати)
param_grid_svd = {
    'n_factors': [50, 100],
    'lr_all': [0.005, 0.010],
    'reg_all': [0.02, 0.05]
}

param_grid_knn = {
    'k': [20, 40],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [False, True]
    }
}

**Обґрунтування вибору параметрів:**

_Поясніть, які параметри впливають на якість моделей і чому обрали саме ці значення._

## 5. Пошук найкращих гіперпараметрів (GridSearchCV)

Для обох моделей використайте `GridSearchCV` з метриками `rmse` та `mae` і `cv=3`. Навчайте об’єкти пошуку **лише на `train_data`**.

In [9]:
# TODO 5.1 — Підберіть гіперпараметри SVD.
# Підказка: GridSearchCV(SVD, param_grid_svd,
#                        measures=['rmse', 'mae'], cv=3, n_jobs=-1)
# Після створення викличте .fit(train_data).

# Ваш код:
gs_svd=GridSearchCV(SVD, param_grid_svd,measures=['rmse', 'mae'], cv=3, n_jobs=-1)
gs_svd.fit(train_data)

In [10]:
print(gs_svd.best_score['rmse'])
print(gs_svd.best_params['rmse'])
print(gs_svd.best_score['mae'])
print(gs_svd.best_params['mae'])

0.9434562000875152
{'n_factors': 100, 'lr_all': 0.01, 'reg_all': 0.05}
0.7447340196205187
{'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}


In [11]:
# TODO 5.2 — Підберіть гіперпараметри KNNBasic.
# Підказка: використайте param_grid_knn та ті самі метрики/cv.

# Ваш код:
gs_knn=GridSearchCV(KNNBasic, param_grid_knn,measures=['rmse', 'mae'], cv=3, n_jobs=-1)
gs_knn.fit(train_data)

In [12]:
print('Best RMSE:', gs_knn.best_score['rmse'])
print('Best params (RMSE):', gs_knn.best_params['rmse'])
print('Best MAE:', gs_knn.best_score['mae'])
print('Best params (MAE):', gs_knn.best_params['mae'])

Best RMSE: 1.0281035692894653
Best params (RMSE): {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}
Best MAE: 0.8153097076252136
Best params (MAE): {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}


In [13]:
# TODO 5.3 — Виведіть найкращий RMSE, MAE та параметри обох моделей.
# Підказки:
# grid_search_svd.best_score['rmse']
# grid_search_svd.best_params['rmse']
# grid_search_knn.best_score['rmse']

# Ваш код:
print('SVD')
print('Best RMSE:', gs_svd.best_score['rmse'])
print('Best MAE: ', gs_svd.best_score['mae'])
print('Params (RMSE):', gs_svd.best_params['rmse'])
print('Params (MAE): ', gs_svd.best_params['mae'])

print('KNN')
print('Best RMSE:', gs_knn.best_score['rmse'])
print('Best MAE: ', gs_knn.best_score['mae'])
print('Params (RMSE):', gs_knn.best_params['rmse'])
print('Params (MAE): ', gs_knn.best_params['mae'])


SVD
Best RMSE: 0.9434562000875152
Best MAE:  0.7447340196205187
Params (RMSE): {'n_factors': 100, 'lr_all': 0.01, 'reg_all': 0.05}
Params (MAE):  {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}
KNN
Best RMSE: 1.0281035692894653
Best MAE:  0.8153097076252136
Params (RMSE): {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}
Params (MAE):  {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}


**Аналіз результатів кросвалідації:**

| Модель | Найкращий CV RMSE | CV MAE | Найкращі параметри за RMSE |
|:--|--:|--:|:--|
| SVD | 0.94 | 0.74 | 'n_factors': 100, 'lr_all': 0.01, 'reg_all': 0.05 |
| KNNBasic | 1.02 | 0.81 | 'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True |

_Порівняйте якість моделей на перехресній перевірці. Зазначте, які параметри стали найкращими._

## 6. Навчання остаточних моделей

Побудуйте `trainset = train_data.build_full_trainset()`. Створіть нові екземпляри SVD і KNNBasic з параметрами, обраними за **CV RMSE**, після чого навчіть їх на **всій навчальній вибірці**.

In [15]:
# TODO 6.1 — Отримайте best_svd_params і best_knn_params за CV RMSE.
# TODO 6.2 — Створіть trainset із train_data.
# TODO 6.3 — Навчіть best_svd і best_knn.
# Для відтворюваності задайте random_state=42 для SVD.

# Ваш код:
best_svd_param=gs_svd.best_params['rmse']
best_knn_param=gs_knn.best_params['rmse']

best_svd = SVD(**best_svd_param)
best_knn = KNNBasic(**best_knn_param)

trainset = train_data.build_full_trainset()

best_svd.fit(trainset)
best_knn.fit(trainset)

Computing the cosine similarity matrix...
Done computing similarity matrix.


## 7. Оцінювання на незалежній тестовій вибірці

Отримайте передбачення для `testset` методами `.test()` обох моделей. Обчисліть `RMSE` та `MAE` за допомогою `surprise.accuracy`.

**Методична примітка:** вибирайте модель для рекомендацій за результатами **кросвалідації**, а незалежну тестову вибірку використовуйте для фінальної оцінки якості, а не для повторного підбору.

In [17]:
# TODO 7.1 — Отримайте svd_predictions та knn_predictions.
# TODO 7.2 — Обчисліть svd_rmse, svd_mae, knn_rmse, knn_mae.
# Підказка: accuracy.rmse(predictions, verbose=False)
#           accuracy.mae(predictions, verbose=False)

# Ваш код:
svd_predictions=best_svd.test(testset)
knn_predictions=best_knn.test(testset)

svd_rmse=accuracy.rmse(svd_predictions, verbose=False)
svd_mae=accuracy.mae(svd_predictions, verbose=False)
knn_rmse=accuracy.rmse(knn_predictions, verbose=False)
knn_mae=accuracy.mae(knn_predictions, verbose=False)

In [18]:
# TODO 7.3 — Сформуйте DataFrame results із колонками:
# Model, RMSE, MAE.
# Відсортуйте за RMSE та покажіть порівняння.

# Ваш код:
results = pd.DataFrame({
    'Model': ['SVD', 'KNNBasic'],
    'RMSE':  [svd_rmse, knn_rmse],
    'MAE':   [svd_mae, knn_mae]
}).sort_values('RMSE').reset_index(drop=True)

results

,Model,RMSE,MAE
0,SVD,0.918991,0.723398
1,KNNBasic,1.011953,0.798421


**Таблиця тестових результатів:**

| Модель | Test RMSE | Test MAE |
|:--|--:|--:|
| SVD | ... | ... |
| KNNBasic | ... | ... |

**Порівняння з CV:** _Прокоментуйте, чи суттєво відрізняються оцінки на кросвалідації та незалежній тестовій вибірці._

Вони відрізняються несуттєво, розбіжність у близько 2 відсотки.

## 8. Топ-10 рекомендацій для користувача

1. Оберіть модель із кращим **CV RMSE**.
2. Задайте `user_id` (вихідний приклад використовує `'196'`).
3. Отримайте фільми, які цей користувач уже оцінив у навчальній вибірці.
4. Для решти фільмів обчисліть прогнозований рейтинг `.predict(uid=..., iid=...).est`.
5. Виведіть **10 фільмів із найбільшим прогнозованим рейтингом**.

**Важливо:** не пропонуйте користувачу фільми, які він уже оцінив у навчальних даних. Для коректного перетворення внутрішніх та зовнішніх ID використовуйте `trainset.to_inner_uid()`, `trainset.to_raw_iid()`.

In [19]:
# Початковий ID можна змінити
user_id = '196'

# TODO 8.1 — Визначте найкращу за CV RMSE модель.
# TODO 8.2 — Перевірте, що користувач присутній у trainset.
# TODO 8.3 — Знайдіть неоцінені ним фільми.
# TODO 8.4 — Спрогнозуйте оцінки та покажіть топ-10.
# Бажані колонки результату: item_id, predicted_rating.
# Передбачте ситуацію, коли обраного user_id немає у trainset.

# Ваш код:
best_model = best_svd if gs_svd.best_score['rmse'] < gs_knn.best_score['rmse'] else best_knn
print('Обрана модель:', type(best_model).__name__)

inner_uid = trainset.to_inner_uid(user_id)

rated_inner_iids = {inner_iid for (inner_iid, _) in trainset.ur[inner_uid]}
rated_raw_iids = {trainset.to_raw_iid(i) for i in rated_inner_iids}
print(f'Користувач {user_id} оцінив у train: {len(rated_raw_iids)} фільмів')

predictions = []
for inner_iid in trainset.all_items():
    if inner_iid in rated_inner_iids:
        continue
    raw_iid = trainset.to_raw_iid(inner_iid)
    est = best_model.predict(uid=user_id, iid=raw_iid).est
    predictions.append((raw_iid, est))

top10 = sorted(predictions, key=lambda x: x[1], reverse=True)[:10]

top10_df = pd.DataFrame(top10, columns=['item_id', 'predicted_rating'])
top10_df

Обрана модель: SVD
Користувач 196 оцінив у train: 29 фільмів


,item_id,predicted_rating
0,318,4.534555
1,64,4.497234
2,479,4.467131
3,114,4.464685
4,199,4.452451
5,187,4.425945
6,313,4.417699
7,483,4.381566
8,320,4.371179
9,480,4.368139


## 9. Перевірка виконання

Позначте після реалізації:

- [ ] Дані `ml-100k` завантажено та перевірено.
- [ ] Є розбиття 80/20 з фіксованим `random_state`.
- [ ] `Reader`, `train_data` і `testset` створені коректно.
- [ ] GridSearchCV для **обох** моделей виконано лише на `train_data`.
- [ ] Для SVD і KNNBasic показано параметри та результати CV.
- [ ] Моделі навчено на повній навчальній частині.
- [ ] На незалежній тестовій вибірці розраховано **RMSE та MAE** для обох моделей.
- [ ] Наведено порівняльну таблицю та інтерпретацію.
- [ ] Сформовано топ-10 рекомендацій без уже оцінених фільмів.
- [ ] Усі комірки виконуються послідовно після перезапуску середовища.

## 10. Висновки

Заповніть **власними результатами**:

1. Яку модель обрано за кросвалідацією і чому?
2. Які найкращі гіперпараметри отримано для обох моделей?
3. Які підсумкові значення RMSE та MAE на тестовій вибірці?
4. Чи є різниця між результатами CV та тестування?
5. Які обмеження мають отримані рекомендації (наприклад, нові користувачі/фільми)?

**Висновок:** _Напишіть 5–8 речень з опорою на отримані числові результати._

---
**Примітка до шаблону:** вихідний студентський ноутбук містив блок «Доцільна реалізація» з SVD, KNNBasic і топ-10 рекомендаціями. Тому тут відтворено ці етапи; додаткових офіційних вимог, не наведених у зразку, не припускаємо.

Ми обрали SVC за найкращий rmse, по якому ми і обирали модель. Різниця між CV та тестом невелика: для svd rmse знизився з 0.9435 до 0.919. MAE з 0.7447 до 0.7234